In [137]:
import clickhouse_connect
import pandas as pd
from pathlib import Path
import os
from dotenv import load_dotenv

In [138]:
load_dotenv()

True

In [139]:
client = clickhouse_connect.get_client(
  host=os.getenv("CLICKHOUSE_HOST"),
  port=int(os.getenv("CLICKHOUSE_PORT")),
  username=os.getenv("CLICKHOUSE_USERNAME"),
  password=str(os.getenv("CLICKHOUSE_PASSWORD")),
  database=os.getenv("CLICKHOUSE_DATABASE"),
)

In [140]:
DATA_DIR = Path("../../data")

In [141]:

def df_to_clickhouse_schema(df: pd.DataFrame, nullable_cols: list = None) -> str:
    nullable_cols = set(nullable_cols or [])

    type_map = {
        'int64': 'Int64',
        'Int64': 'Int64',
        'int32': 'Int32',
        'Int32': 'Int32',
        'uint64': 'UInt64',
        'uint32': 'UInt32',
        'float64': 'Float64',
        'Float64': 'Float64',
        'float32': 'Float32',
        'bool': 'Bool',
        'boolean': 'Bool',
        'object': 'String',
        'string': 'String',
        'category': 'LowCardinality(String)',
        'datetime64[ns]': 'DateTime64(3)',
    }

    columns = []
    for col, dtype in df.dtypes.items():
        dtype_str = str(dtype)
        # Handle timezone-aware datetimes
        if dtype_str.startswith('datetime64'):
            ch_type = 'DateTime64(3)'
        else:
            ch_type = type_map.get(dtype_str, 'String')

        # Wrap in Nullable if the column has NaNs or is specified
        if col in nullable_cols or df[col].isna().any():
            if not ch_type.startswith('LowCardinality'):
                ch_type = f'Nullable({ch_type})'

        columns.append(f"{col} {ch_type}")

    return ",\n".join(columns)

In [142]:
def extract_to_clickhouse(path: Path):
  try:
    dataset_file = path
    df = pd.read_csv(dataset_file, sep=',')

    table_name = path.stem.removeprefix('olist_').removesuffix('_dataset').lower()
    schema = df_to_clickhouse_schema(df)

    client.command(f"DROP TABLE IF EXISTS {table_name}")

    client.command(f"""
    CREATE TABLE {table_name} (
      {schema}
    ) ENGINE = MergeTree()
    """)

    client.insert_df(table_name, df)

    print(f"{path.name} successfully extracted to ClickHouse in {table_name} table")
  except Exception as e:
    print(f"Error extracting dataset to ClickHouse: {e}")

In [143]:
for dataset in sorted(DATA_DIR.glob("*.csv")):
  extract_to_clickhouse(dataset)

olist_customers_dataset.csv successfully extracted to ClickHouse in customers table
olist_geolocation_dataset.csv successfully extracted to ClickHouse in geolocation table
olist_order_items_dataset.csv successfully extracted to ClickHouse in order_items table
olist_order_payments_dataset.csv successfully extracted to ClickHouse in order_payments table
olist_order_reviews_dataset.csv successfully extracted to ClickHouse in order_reviews table
olist_orders_dataset.csv successfully extracted to ClickHouse in orders table
olist_products_dataset.csv successfully extracted to ClickHouse in products table
olist_sellers_dataset.csv successfully extracted to ClickHouse in sellers table
product_category_name_translation.csv successfully extracted to ClickHouse in product_category_name_translation table


In [144]:
tables = client.query("SHOW TABLES").result_columns[0]

for table in tables:
  print(f"\n=================== Sample Data of {table} table ===================\n")
  print(client.query_df(f"SELECT * FROM {table} LIMIT 3"))


=================== Sample Data of customers table ===================

                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  

=================== Sample Data of geolocation table ===================

   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1037       -23.545621       -46.639292   
1                         1046       -23.546081       -46.644820   
2                         1046       -23.546129       -46.642951   

  geolocation_